# TP 3 — Sélection d'estimateurs et diagnostics

**Objectifs**

- Comparer plusieurs estimateurs par validation croisée répétée.
- Lire un boxplot de scores pour décider entre modèles.
- Tracer une **courbe d'apprentissage** et diagnostiquer biais / variance.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

data = load_breast_cancer()
X, y = data.data, data.target

## Exercice 1 — Comparer 4 modèles par CV répétée

1. Construisez 4 estimateurs :
   - `"logreg"` : Pipeline `StandardScaler + LogisticRegression(max_iter=2000)`
   - `"svm"`    : Pipeline `StandardScaler + SVC(kernel="rbf")`
   - `"rf"`     : `RandomForestClassifier(n_estimators=200, random_state=0)`
   - `"gbm"`    : `GradientBoostingClassifier(random_state=0)`
2. Évaluez chacun via `RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)` et `cross_val_score(scoring="f1")`.
3. Tracez un boxplot des 4 distributions de scores.

<details>
<summary>💡 Coup de pouce — comparer 4 modèles par CV répétée</summary>

**🎯 But :** obtenir une comparaison **fiable** entre modèles en lissant la variance des splits CV.

**RepeatedStratifiedKFold**

```python
from sklearn.model_selection import RepeatedStratifiedKFold
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
```

Total = **15 folds** (5 splits × 3 répétitions, chaque répétition avec un mélange différent). Avantages :
- Stratification : préserve les proportions de classes.
- Répétition : 3 estimations indépendantes → statistique plus robuste qu'une seule CV à 5 folds.

**Évaluer chaque modèle**

```python
from sklearn.model_selection import cross_val_score
scores_logreg = cross_val_score(model_logreg, X, y, cv=cv, scoring='f1')
scores_svm    = cross_val_score(model_svm,    X, y, cv=cv, scoring='f1')
scores_rf     = cross_val_score(model_rf,     X, y, cv=cv, scoring='f1')
scores_gbm    = cross_val_score(model_gbm,    X, y, cv=cv, scoring='f1')
```

Chaque tableau a **15 valeurs**.

**Pourquoi `scoring='f1'` ?**

`f1` = moyenne harmonique précision/rappel. Plus robuste que l'accuracy sur datasets déséquilibrés. Choisir selon le problème :
- `'accuracy'` : datasets équilibrés.
- `'f1'`, `'f1_macro'`, `'f1_weighted'` : déséquilibrés.
- `'roc_auc'` : pour ranking.

**Boxplot comparatif**

```python
plt.boxplot([scores_logreg, scores_svm, scores_rf, scores_gbm],
            labels=['logreg', 'svm', 'rf', 'gbm'])
plt.ylabel('F1 score')
plt.title('Comparaison 5×3 CV répétée')
```

Le boxplot montre :
- **Médiane** : performance typique.
- **Boîte** : 50 % central des scores.
- **Moustaches** : variabilité.
- **Outliers** : folds particulièrement difficiles/faciles.

→ Si les boîtes se **chevauchent**, les modèles sont **indistinguables au niveau de bruit**. Pour conclure proprement, faire un test de Wilcoxon apparié.

</details>

In [2]:
# TODO


## Exercice 2 — Courbe d'apprentissage du meilleur modèle

Sur le modèle dont la médiane F1 est la plus haute :

1. Utilisez `learning_curve` avec `train_sizes=np.linspace(0.1, 1.0, 8)` et `cv=5`.
2. Tracez les courbes train (moyenne ± std) et validation (moyenne ± std) en fonction de la taille du train.
3. Commentez le diagnostic :
   - Si gap train/val très large → sur-apprentissage.
   - Si train et val faibles tous les deux → sous-apprentissage.
   - Si val croit encore → plus de données pourrait aider.

<details>
<summary>💡 Coup de pouce — courbe d'apprentissage</summary>

**🎯 But :** voir si le modèle a besoin de plus de données (régime sous-apprentissage) ou s'il sature (a déjà tout extrait).

**Tracer la courbe**

```python
from sklearn.model_selection import learning_curve
import numpy as np

sizes, train_scores, val_scores = learning_curve(
    model, X, y,
    train_sizes=np.linspace(0.1, 1.0, 8),    # 8 tailles : 10%, 22%, 35%, ..., 100%
    cv=5,
    scoring='f1',
    n_jobs=-1,
)
```

`train_scores` et `val_scores` ont shape **(8, 5)** : 8 tailles × 5 folds.

**Moyenne et écart-type**

```python
train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)
```

`axis=1` = on moyenne sur les 5 folds pour chaque taille.

**Tracé avec bandes de confiance**

```python
plt.plot(sizes, train_mean, 'o-', label='train')
plt.fill_between(sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)
plt.plot(sizes, val_mean, 's-', label='validation')
plt.fill_between(sizes, val_mean - val_std, val_mean + val_std, alpha=0.2)
plt.xlabel('Taille du train'); plt.ylabel('F1')
plt.legend()
```

`fill_between(x, y_low, y_high, alpha=0.2)` crée une bande translucide qui montre la variabilité — beaucoup plus parlant que des barres d'erreur.

**Lire la courbe**

| Pattern | Diagnostic | Action |
|---|---|---|
| train et val **convergent en haut** | Bonne situation | Garder le modèle, peut-être plus de capacité |
| train **haut** et val **stagne bas** | Overfitting | Plus de données, régularisation, modèle plus simple |
| Les deux **stagnent bas** | Underfitting | Modèle plus puissant, features additionnelles |
| val **monte encore** à 100 % train | Plus de données aide | Collecter plus, augmentation |

</details>

In [3]:
# TODO


## Exercice 3 — Apprentissage d'un modèle volontairement sous-régularisé

Tracez la courbe d'apprentissage d'un `DecisionTreeClassifier` sans limite de profondeur (`max_depth=None`). Que constatez-vous sur l'écart train/val ? Comment réduiriez-vous le sur-apprentissage ?

<details>
<summary>💡 Coup de pouce — un modèle volontairement sous-régularisé</summary>

**🎯 But :** voir concrètement à quoi ressemble l'overfitting et comprendre les leviers pour le combattre.

**Forcer l'overfitting**

```python
from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier(max_depth=None, random_state=0)
tree.fit(X_train, y_train)
print(f"Train : {tree.score(X_train, y_train):.3f}")
print(f"Test  : {tree.score(X_test,  y_test):.3f}")
```

Avec `max_depth=None`, l'arbre grandit jusqu'à parfaitement classer chaque échantillon du train → train accuracy = **1.0** (mémorisation). Le test sera bien plus bas → écart énorme.

**Pourquoi ça overfit**

Un arbre sans contrainte peut « apprendre » des règles ultra-spécifiques qui collent au bruit individuel de chaque échantillon. Ces règles ne généralisent pas.

**Les leviers pour réduire l'overfit**

1. **Limiter la profondeur** :
   ```python
   tree = DecisionTreeClassifier(max_depth=5)
   ```
   Force l'arbre à rester simple, à ne capturer que les règles « grosses ».

2. **Augmenter le min_samples_split** :
   ```python
   tree = DecisionTreeClassifier(min_samples_split=20)
   ```
   Empêche les feuilles avec peu d'échantillons (souvent du sur-apprentissage de bruit).

3. **Ensemble de modèles** :
   ```python
   from sklearn.ensemble import RandomForestClassifier
   rf = RandomForestClassifier(n_estimators=100, max_depth=None)
   ```
   La forêt aléatoire moyenne les prédictions de 100 arbres → variance réduite **par construction**, même si chaque arbre overfit.

4. **Plus de données** : la solution la plus durable. Mais souvent indisponible.

5. **Régularisation explicite** : pour les modèles linéaires (`C` plus petit en LogReg, `alpha` plus grand en Ridge), pour les NN (dropout, weight decay).

**Validation sur la courbe d'apprentissage de l'Exo 2**

Reprenez la courbe d'apprentissage avec ce modèle overfit : vous verrez le train à 1.0 dès le début et la validation très en dessous → la signature de l'overfit.

</details>

In [4]:
# TODO
